In [1]:
import polars as pl 
from pathlib import Path 
import logging
import json
import time
import requests

logger = logging.getLogger(__name__)

STEP_X3_OUTPUT_PATH = Path("/cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/t11_RM_TM_updated_CF_pipeline/cf_pdb_structure_similarity/aggregate_cf_for_pdb_eval_with_combfold_results_paths.parquet") 
REFERENCE_PDB_DIR = Path("/cluster/project/beltrao/kdammer/master_thesis/data/reference_pdb")

In [2]:
df_stepX3 = pl.read_parquet(STEP_X3_OUTPUT_PATH)

In [3]:
print(df_stepX3.columns)

['complex_ac', 'identifiers', 'pred_1', 'pred_2', 'pred_3', 'correct_pred_count', 'correct_pred_rank', 'CF_1_n_assemblies', 'CF_1_confidence', 'CF_1_reason', 'CF_2_n_assemblies', 'CF_2_confidence', 'CF_2_reason', 'CF_3_n_assemblies', 'CF_3_confidence', 'CF_3_reason', 'CF_true_n_assemblies', 'CF_true_confidence', 'CF_true_reason', 'combfold_job_ids', 'n_pairs_used', 'pairs_used', 'n_proteins', 'pdb_id', 'match_class', 'pdb_stoichiometry', 'CP_stochiometry', 'reference_pdb_model', 'found_match_reference_pdb_model', 'Combfold_result_path', 'n_combfold_outputs']


In [4]:
assert df_stepX3["found_match_reference_pdb_model"].all(), "Not all rows have a matching reference PDB model. Please check the data. This should have been filtered in the previous step X3."
df_relevant_cols = df_stepX3[["complex_ac", "identifiers", "n_proteins", "pdb_id", "reference_pdb_model", "CP_stochiometry", "Combfold_result_path", "n_pairs_used", "pairs_used", "correct_pred_rank", "n_combfold_outputs"]]

In [5]:
def _get_reference_pdb_path(pdb_id: str, reference_pdb_model: str, reference_pdb_dir: Path = REFERENCE_PDB_DIR) -> Path:
    """
    Given a PDB ID and a reference PDB model, return the path to the corresponding reference PDB file.
    """
    pdb_id = pdb_id.lower()
    if reference_pdb_model == "au":
        file_path_without_suffix =  (reference_pdb_dir / pdb_id / pdb_id)
    else:
        file_path_without_suffix = (reference_pdb_dir / pdb_id / f"{pdb_id}-assembly{reference_pdb_model}")

    if file_path_without_suffix.with_suffix(".pdb").exists():
        return file_path_without_suffix.with_suffix(".pdb")
    elif file_path_without_suffix.with_suffix(".cif").exists():
        return file_path_without_suffix.with_suffix(".cif")
    raise FileNotFoundError(f"Neither .pdb nor .cif file found for {pdb_id} at {file_path_without_suffix}.")

# resolve reference pdb path per row 
df_relevant_cols = df_relevant_cols.with_columns(
    pl.struct(["pdb_id", "reference_pdb_model"])
    .map_elements(
        lambda row: str(_get_reference_pdb_path(row["pdb_id"], row["reference_pdb_model"])),
        return_dtype=pl.Utf8,
    )
    .alias("reference_pdb_path")
)

In [6]:
def _get_combfold_output_paths(combfold_result_path: str, n_combfold_outputs: int) -> list[Path]:
    """
    Given a Combfold_result_path, list the assembled CombFold output files under
    `{combfold_result_path}/assembled_results/`, sorted for a stable/reproducible order.

    Raises if the number of files found does not match `n_combfold_outputs` -- we don't want
    to silently proceed with a mismatched count.
    """
    combfold_result_path = Path(combfold_result_path)
    if n_combfold_outputs == 0:
        return []

    assembled_results_dir = combfold_result_path / "assembled_results"

    if not assembled_results_dir.is_dir():
        raise FileNotFoundError(f"assembled_results dir does not exist: {assembled_results_dir}")

    assert all(p.suffix in (".txt", ".pdb") for p in assembled_results_dir.iterdir()), f"Unexpected file extension found in {assembled_results_dir}. Expected only .txt or .pdb files."
    output_paths = sorted(assembled_results_dir.glob("*.pdb"))

    if len(output_paths) != n_combfold_outputs:
        raise ValueError(
            f"Expected {n_combfold_outputs} CombFold outputs in {assembled_results_dir}, "
            f"found {len(output_paths)}."
        )

    return output_paths

# %%
# resolve the list of CombFold output paths per row, then explode into one row per output
df_relevant_cols = df_relevant_cols.with_columns(
    pl.struct(["Combfold_result_path", "n_combfold_outputs"])
    .map_elements(
        lambda row: [
            str(p)
            for p in _get_combfold_output_paths(row["Combfold_result_path"], row["n_combfold_outputs"])
        ],
        return_dtype=pl.List(pl.Utf8),
    )
    .alias("combfold_output_path")
)

df_long = df_relevant_cols.explode("combfold_output_path")

# sanity checks: row count should scale by n_combfold_outputs (treating 0 -> 1 row from explode),
# and only rows with n_combfold_outputs == 0 should have a null combfold_output_path
expected_n_rows = (
    df_relevant_cols["n_combfold_outputs"].clip(lower_bound=1).sum()
)
assert df_long.height == expected_n_rows, (
    f"Expected {expected_n_rows} rows after exploding, got {df_long.height}."
)

expected_null_outputs = (df_relevant_cols["n_combfold_outputs"] == 0).sum()
actual_null_outputs = df_long["combfold_output_path"].null_count()
assert actual_null_outputs == expected_null_outputs, (
    f"Expected {expected_null_outputs} null combfold_output_path rows (n_combfold_outputs == 0), "
    f"got {actual_null_outputs}."
)

assert df_long["reference_pdb_path"].null_count() == 0, "Found null reference_pdb_path."

logger.info(f"Built long df with {df_long.height} rows from {df_relevant_cols.height} complexes.")


In [7]:
# get sifts

SIFTS_FILENAME_TEMPLATE = "{pdb_id}_sifts.json"
SIFTS_URL_TEMPLATE = "https://www.ebi.ac.uk/pdbe/api/mappings/uniprot/{pdb_id}"


def _sifts_path(pdb_id: str) -> Path:
    return REFERENCE_PDB_DIR / pdb_id / SIFTS_FILENAME_TEMPLATE.format(pdb_id=pdb_id)


def _download_sifts(pdb_id: str, timeout: int = 15) -> dict | None:
    url = SIFTS_URL_TEMPLATE.format(pdb_id=pdb_id)
    try:
        resp = requests.get(url, timeout=timeout)
    except requests.RequestException as exc:
        logger.warning(f"{pdb_id}: SIFTS request failed ({exc}) — skipping")
        return None

    if resp.status_code == 404:
        logger.warning(f"{pdb_id}: SIFTS API returned 404 (no UniProt mapping) — skipping")
        return None
    if resp.status_code != 200:
        logger.warning(f"{pdb_id}: SIFTS API returned status {resp.status_code} — skipping")
        return None

    try:
        return resp.json()
    except ValueError:
        logger.warning(f"{pdb_id}: SIFTS response was not valid JSON — skipping")
        return None


def ensure_sifts_files(df, pdb_id_col: str = "pdb_id", sleep_between_requests: float = 0.2):
    """For every unique pdb_id in df, ensure a cached SIFTS UniProt-mapping file
    exists under REFERENCE_PDB_DIR/{pdb_id}/. Downloads if missing.

    Returns dict: pdb_id -> bool (True if file present/available after this call).
    """
    unique_pdb_ids = sorted({str(pid).lower() for pid in df[pdb_id_col].unique()})
    logger.info(f"Checking SIFTS UniProt mappings for {len(unique_pdb_ids)} unique PDB IDs")

    status = {}
    for pdb_id in unique_pdb_ids:
        pdb_dir = REFERENCE_PDB_DIR / pdb_id
        if not pdb_dir.is_dir():
            logger.warning(f"{pdb_id}: no reference directory {pdb_dir} — skipping SIFTS check")
            status[pdb_id] = False
            continue

        sifts_path = _sifts_path(pdb_id)
        if sifts_path.exists():
            logger.debug(f"{pdb_id}: SIFTS file already present at {sifts_path}")
            status[pdb_id] = True
            continue

        logger.info(f"{pdb_id}: SIFTS file missing — downloading")
        data = _download_sifts(pdb_id)
        if data is None:
            status[pdb_id] = False
            continue

        sifts_path.write_text(json.dumps(data))
        logger.info(f"{pdb_id}: SIFTS file written to {sifts_path}")
        status[pdb_id] = True

        time.sleep(sleep_between_requests)  # be polite to the EBI API

    n_ok = sum(status.values())
    n_missing = len(status) - n_ok
    if n_missing:
        logger.warning(f"SIFTS mapping unavailable for {n_missing}/{len(status)} PDB IDs: "
                        f"{[k for k, v in status.items() if not v]}")

    return status

In [8]:
#download sifsts, if it doent exist already
ensure_sifts_files(df_long)

{'1x3z': True,
 '2p22': True,
 '2qlv': True,
 '2qsf': True,
 '4dgw': True,
 '5dfz': True,
 '8qtn': True}

In [10]:
#TODO handle complexes with stoi bigger 2 
df_long = df_long.filter(pl.col("complex_ac") != "CPX-1706")

In [ ]:
import json, re
from Bio.PDB import MMCIFParser, PDBParser


def _get_structure_chain_ids(struct_path: str) -> set[str]:
    parser = MMCIFParser(QUIET=True) if struct_path.endswith(".cif") else PDBParser(QUIET=True)
    structure = parser.get_structure("x", struct_path)
    return {chain.id for chain in structure[0]}


def _build_chain_mapping(
    reference_pdb_path: str, combfold_output_path: str | None, complex_ac: str | None = None
) -> dict[str, tuple[str, str]]:
    if combfold_output_path is None:
        logger.warning(f"{complex_ac or '?'}: no combfold_output_path (n_combfold_outputs == 0) -- empty mapping.")
        return {}

    pdb_dir = Path(reference_pdb_path).parent
    sifts = json.loads((pdb_dir / SIFTS_FILENAME_TEMPLATE.format(pdb_id=pdb_dir.name)).read_text())
    unp_entries = sifts[pdb_dir.name]["UniProt"]

    assembly_chain_ids = _get_structure_chain_ids(reference_pdb_path)
    print("unp_entries", unp_entries)
    print("assembly_chain_ids", assembly_chain_ids)

    pdb_chains = {}
    for unp, e in unp_entries.items():
        print(unp, e)
        chain_ids = {m["chain_id"] for m in e["mappings"]}
        chain_ids_in_assembly = chain_ids & assembly_chain_ids

        if len(chain_ids_in_assembly) != 1:
            logger.warning(
                f"{complex_ac or '?'}: {pdb_dir.name}/{unp} -- SIFTS chain_ids {chain_ids}, "
                f"of which {chain_ids_in_assembly} present in assembly file {reference_pdb_path}. "
                f"Expected exactly 1 -- empty mapping."
            )
            return {}
        pdb_chains[unp] = next(iter(chain_ids_in_assembly))

    chain_list = Path(combfold_output_path).parent.parent / "_unified_representation" / "assembly_output" / "chain.list"
    cf_chains = dict(re.match(r"^(\w+)_(\w+)\.pdb$", line).groups() for line in chain_list.read_text().split())
    if set(cf_chains) != set(pdb_chains):
        logger.warning(
            f"{complex_ac or '?'}: uniprot mismatch cf vs pdb: {set(cf_chains) ^ set(pdb_chains)} -- empty mapping."
        )
        return {}

    return {unp: (cf_chains[unp], pdb_chains[unp]) for unp in cf_chains}


# df_long = df_long.with_columns(
#     pl.struct(["reference_pdb_path", "combfold_output_path", "complex_ac"])
#     .map_elements(
#         lambda r: _build_chain_mapping(r["reference_pdb_path"], r["combfold_output_path"], r["complex_ac"]),
#         return_dtype=pl.Object,
#     )
#     .alias("chain_mapping")
# )

In [41]:
idx = 1
_build_chain_mapping(df_long["reference_pdb_path"][idx], df_long["combfold_output_path"][idx], df_long["complex_ac"][idx])

unp_entries {'Q02890': {'name': 'PNG1_YEAST', 'mappings': [{'entity_id': 1, 'chain_id': 'A', 'struct_asym_id': 'A', 'unp_start': 8, 'unp_end': 342, 'start': {'author_residue_number': 8, 'author_insertion_code': '', 'residue_number': 1}, 'end': {'author_residue_number': None, 'author_insertion_code': '', 'residue_number': 335}, 'identity': 1.0, 'coverage': 0.923}], 'identifier': 'PNG1_YEAST'}, 'P32628': {'name': 'RAD23_YEAST', 'mappings': [{'entity_id': 2, 'chain_id': 'B', 'struct_asym_id': 'B', 'unp_start': 238, 'unp_end': 309, 'start': {'author_residue_number': None, 'author_insertion_code': '', 'residue_number': 1}, 'end': {'author_residue_number': 309, 'author_insertion_code': '', 'residue_number': 72}, 'identity': 1.0, 'coverage': 0.181}], 'identifier': 'RAD23_YEAST'}}
assembly_chain_ids {'A', 'B', 'C', 'D', 'I'}
Q02890 {'name': 'PNG1_YEAST', 'mappings': [{'entity_id': 1, 'chain_id': 'A', 'struct_asym_id': 'A', 'unp_start': 8, 'unp_end': 342, 'start': {'author_residue_number': 8, '

{'P32628': ('A', 'B'), 'Q02890': ('B', 'A')}

In [12]:
import subprocess

USALIGN_BIN = "/cluster/project/beltrao/kdammer/master_thesis/tools/usalign/USalign/USalign"

def run_usalign_mm(structure1: str, structure2: str, mm: int = 1, ter: int = 0) -> str:
    cmd = [USALIGN_BIN, structure1, structure2,
           "-mm", str(mm), "-ter", str(ter), "-mol", "prot"]
    usalign = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return usalign.stdout

#TM score?

In [13]:
"""
def run_usalign_mm(structure1: str, structure2: str, mm: int = 1, ter: int = 0, TMscore: int = 2) -> str:
    cmd = [USALIGN_BIN, structure1, structure2, "-mm", str(mm), "-ter", str(ter), "-TMscore", str(TMscore)]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return result.stdout
"""

'\ndef run_usalign_mm(structure1: str, structure2: str, mm: int = 1, ter: int = 0, TMscore: int = 2) -> str:\n    cmd = [USALIGN_BIN, structure1, structure2, "-mm", str(mm), "-ter", str(ter), "-TMscore", str(TMscore)]\n    result = subprocess.run(cmd, capture_output=True, text=True, check=True)\n    return result.stdout\n'

In [14]:
usalign_result = run_usalign_mm("/cluster/project/beltrao/kdammer/master_thesis/data/reference_pdb/5dfz/5dfz-assembly1.cif", "/cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/t11_RM_TM_updated_CF_pipeline/CombFold/P22219x1_P22543x1_Q02948x1_Q05919x1_pool_output/assembled_results/output_clustered_0.pdb")

In [15]:
print(usalign_result)


 ********************************************************************
 * US-align (Version 20260813)                                      *
 * Universal Structure Alignment of Proteins and Nucleic Acids      *
 * Reference: C Zhang, L Freddolino, Y Zhang. (2026) Nat Protoc     *
 *            C Zhang, M Shine, AM Pyle, Y Zhang. (2022) Nat Methods*
 *            C Zhang, AM Pyle (2022) iScience.                     *
 * Please email comments and suggestions to zhang@zhanggroup.org    *
 ********************************************************************

Name of Structure_1: /cluster/project/beltrao/kdammer/master_thesis/data/reference_pdb/5dfz/5dfz-assembly1.cif:1,A:1,B:1,D:1,C:1,E:1,G: (to be superimposed onto Structure_2)
Name of Structure_2: /cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/t11_RM_TM_updated_CF_pipeline/CombFold/P22219x1_P22543x1_Q02948x1_Q05919x1_pool_output/assembled_results/output_clustered_0.pdb:1,C:1,A:1,D::::1,B
Length of Structure_1: 2853 residue

In [16]:
def parse_usalign_per_chain(stdout: str) -> list[dict]:
    """Extract per-chain RMSD/TM-score blocks from US-align -mm 1 output."""
    blocks = re.split(r"(?=Name of Chain_1:)", stdout)
    results = []
    for block in blocks:
        chain1_match = re.search(r"Name of Chain_1:\s*(\S+)", block)
        chain2_match = re.search(r"Name of Chain_2:\s*(\S+)", block)
        rmsd_match = re.search(r"RMSD=\s*([\d.]+)", block)
        tm1_match = re.search(r"TM-score=\s*([\d.]+)\s*\(normalized by length of Chain_1\)", block)
        tm2_match = re.search(r"TM-score=\s*([\d.]+)\s*\(normalized by length of Chain_2\)", block)

        if chain1_match and rmsd_match:
            results.append({
                "chain1": chain1_match.group(1),
                "chain2": chain2_match.group(1) if chain2_match else None,
                "rmsd": float(rmsd_match.group(1)),
                "tm_score_1": float(tm1_match.group(1)) if tm1_match else None,
                "tm_score_2": float(tm2_match.group(1)) if tm2_match else None,
            })
    return results

per_chain = parse_usalign_per_chain(usalign_result)
for entry in per_chain:
    print(entry)

In [17]:
type(result.stdout)

NameError: name 'result' is not defined

was the pdb in stoic?


In [ ]:
stoic_data = pl.read_csv("/cluster/project/beltrao/kdammer/master_thesis/data/Stoic/data_file_stoic.csv")

pdb_uniprot_mapping = pl.read_csv(
    "/cluster/project/beltrao/kdammer/master_thesis/data/pdb/pdb_chain_uniprot.csv",
    skip_rows=1,
    schema_overrides={
        "RES_BEG": pl.Utf8,
        "RES_END": pl.Utf8,
        "PDB_BEG": pl.Utf8,
        "PDB_END": pl.Utf8,
        "SP_BEG": pl.Utf8,
        "SP_END": pl.Utf8,
    },
)


# is exactly this pdb in stoic
df_long = df_long.with_columns(
    eaxct_pdb_in_stoic=pl.col("pdb_id").str.to_lowercase().is_in(
        stoic_training["pdb_id"].str.to_lowercase().implode()
    )
)


#is any pdb with these proteins in stoic training 
pdb_uniprot_mapping = pdb_uniprot_mapping.rename(
    {c: c.lower() for c in pdb_uniprot_mapping.columns}
)

pdb_to_proteins = (
    pdb_uniprot_mapping
    .group_by("pdb")
    .agg(
        pl.col("sp_primary").unique().alias("proteins"),
        pl.col("chain").n_unique().alias("n_chains"),
    )
)


stoic_data = stoic_data.select(["pdb_id","split"])

stoic_data = stoic_data.join(
    pdb_to_proteins,
    left_on="pdb_id",
    right_on="pdb",
    how="left",
)

stoic_training = stoic_data.filter(pl.col("split") == "train")

In [ ]:
df_long = df_long.with_columns(
    pl.col("CP_stochiometry")
    .map_elements(lambda s: list(json.loads(s).keys()), return_dtype=pl.List(pl.Utf8))
    .alias("CP_stochiometry_protlist")
)

# canonicalize: sort list elements, join into a single string key
def canon(x):
    return x.list.sort().list.join(",")

in_stoic_training = set(canon(stoic_training["proteins"]).to_list())

df_long = df_long.with_columns(
    canon(pl.col("CP_stochiometry_protlist"))
).with_columns(
    pl.col("CP_stochiometry_protlist").is_in(in_stoic_training).alias("pdb_with_identical_proteins_set_to_CF_in_stoic")
).drop("CP_stochiometry_protlist")

In [ ]:
from Bio.PDB import PDBParser, MMCIFParser, Superimposer
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1
import numpy as np

def _get_chain_ca(struct_path: str, chain_id: str):
    """Returns (sequence_str, list[CA Atom]) for one chain, standard amino acids only."""
    parser = MMCIFParser(QUIET=True) if struct_path.endswith(".cif") else PDBParser(QUIET=True)
    structure = parser.get_structure("x", struct_path)
    chain = structure[0][chain_id]

    residues = [r for r in chain if is_aa(r, standard=True) and "CA" in r]
    seq = "".join(seq1(r.get_resname()) for r in residues)
    ca_atoms = [r["CA"] for r in residues]
    return seq, ca_atoms


def _aligned_ca_pairs(seq_cf, ca_cf, seq_pdb, ca_pdb):
    aligner = PairwiseAligner()
    aligner.mode = "global"
    aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")  
    aligner.open_gap_score = -10                                          
    aligner.extend_gap_score = -0.5                                       
    alignment = aligner.align(seq_cf, seq_pdb)[0]
    aligned_cf, aligned_pdb = alignment.indices

    matched_cf, matched_pdb, n_identical = [], [], 0
    for i_cf, i_pdb in zip(aligned_cf, aligned_pdb):
        if i_cf == -1 or i_pdb == -1:
            continue
        matched_cf.append(ca_cf[i_cf])
        matched_pdb.append(ca_pdb[i_pdb])
        n_identical += seq_cf[i_cf] == seq_pdb[i_pdb]

    identity = n_identical / len(matched_cf) if matched_cf else 0.0   # also guards the divide-by-zero
    return matched_cf, matched_pdb, identity

def compute_rmsd(combfold_output_path: str, reference_pdb_path: str, chain_mapping: dict[str, tuple[str, str]]):
    """chain_mapping: {uniprot: (cf_chain_id, pdb_chain_id)}. Returns (rmsd, min_identity)."""
    all_cf_atoms, all_pdb_atoms, identities = [], [], []

    for uniprot, (cf_chain, pdb_chain) in chain_mapping.items():
        seq_cf, ca_cf = _get_chain_ca(combfold_output_path, cf_chain)
        seq_pdb, ca_pdb = _get_chain_ca(reference_pdb_path, pdb_chain)
        assert seq_cf and seq_pdb, f"{uniprot}: empty sequence (cf={len(seq_cf)}, pdb={len(seq_pdb)})"

        matched_cf, matched_pdb, identity = _aligned_ca_pairs(seq_cf, ca_cf, seq_pdb, ca_pdb)
        assert matched_cf, f"{uniprot}: no aligned residues between CF chain {cf_chain} and PDB chain {pdb_chain}"

        all_cf_atoms += matched_cf
        all_pdb_atoms += matched_pdb
        identities.append(identity)

    sup = Superimposer()
    sup.set_atoms(all_pdb_atoms, all_cf_atoms)  # fixed=pdb, moving=cf
    return sup.rms, min(identities)